# Assignment — RSA: Models vs. STG

Reusable RSA toolbox carried over from the practical (numpy / scipy / matplotlib only), plus a skeleton for the assignment.

RSA recap: each system → `(n_items, n_features)` → RDM `(n_items, n_items)` → correlate the **upper triangles (no diagonal)** of the RDMs.

In [1]:
%pip install torch torchaudio soundfile --q

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.set_printoptions(precision=3, suppress=True)

DATA_DIR = Path("data")                    # adjust to where you unpacked the data
MODEL_DIR = Path("assignment") / "models"  # adjust to where the checkpoints live

## RSA toolbox

In [3]:
def compute_rdm(patterns: np.ndarray, metric: str = "euclidean") -> np.ndarray:
    """RDM from patterns of shape (n_items, ...). Extra dims (e.g. conv channels x freq x time)
    are flattened. metric = "euclidean" | "correlation"."""
    patterns = np.asarray(patterns, dtype=float).reshape(len(patterns), -1)

    if metric == "euclidean":
        sq = np.sum(patterns ** 2, axis=1)
        rdm = np.sqrt(np.maximum(sq[:, None] + sq[None, :] - 2 * patterns @ patterns.T, 0))
        np.fill_diagonal(rdm, 0)
        return rdm

    if metric == "correlation":
        rdm = 1 - np.corrcoef(patterns)
        np.fill_diagonal(rdm, 0)
        return rdm

    raise ValueError(f"Unknown metric: {metric}")


def upper_triangle(rdm: np.ndarray) -> np.ndarray:
    """Off-diagonal upper-triangular entries as a 1D vector."""
    i, j = np.triu_indices(rdm.shape[0], k=1)
    return rdm[i, j]


def compare_rdms(rdm_a: np.ndarray, rdm_b: np.ndarray, method: str = "pearson") -> float:
    """Correlation between the upper triangles of two RDMs. method = "pearson" | "spearman"."""
    a, b = upper_triangle(rdm_a), upper_triangle(rdm_b)
    if method == "pearson":
        return stats.pearsonr(a, b)[0]
    if method == "spearman":
        return stats.spearmanr(a, b)[0]
    raise ValueError(f"Unknown method: {method}")


def compare_to_many(reference_rdm, candidate_rdms: dict, method="pearson") -> list:
    """Returns [(name, score), ...] sorted best -> worst."""
    results = [(name, compare_rdms(reference_rdm, rdm, method)) for name, rdm in candidate_rdms.items()]
    return sorted(results, key=lambda t: t[1], reverse=True)

In [4]:
def mean_ci(scores, confidence=0.95):
    """Returns (mean, ci_low, ci_high) using the t-distribution.
    (util.mean_and_ci does the same job if you prefer the provided helper.)"""
    scores = np.asarray(scores, dtype=float)
    m = scores.mean()
    h = stats.sem(scores) * stats.t.ppf((1 + confidence) / 2, df=len(scores) - 1)
    return m, m - h, m + h

In [5]:
def plot_rdm(rdm, labels=None, title="", ax=None):
    """Heatmap of one RDM. Returns the image object (needed for the colorbar)."""
    if ax is None:
        _, ax = plt.subplots()
    im = ax.imshow(rdm, cmap="viridis")
    if labels is not None:
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=90)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels)
    ax.set_title(title)
    return im


def plot_rdms(rdms: dict, labels=None):
    """Several RDMs side by side. rdms = {"name": rdm}"""
    fig, axes = plt.subplots(1, len(rdms), figsize=(4 * len(rdms), 4), squeeze=False)
    for ax, (name, rdm) in zip(axes[0], rdms.items()):
        im = plot_rdm(rdm, labels=labels, title=name, ax=ax)
        fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

## Setup — data and models

Check `core.py` for the exact return formats of these helpers before relying on them.

In [6]:
from core import SantoroDataset, MODEL_CLASSES, extract_activations, load_yamnet_activations

dataset = SantoroDataset()
# TODO: build a DataLoader over `dataset` (no shuffling, so stimulus order stays fixed)
# TODO: pull out the STG activity -> rdm_brain = compute_rdm(stg, metric)

## Part I — RSA: models vs. brain

1. **Untrained models**: `MODEL_CLASSES[name](num_classes=50)`, no checkpoint → per-layer RDMs
2. **Trained models**: same, then load weights from `MODEL_DIR` → per-layer RDMs
3. **YAMNet**: `load_yamnet_activations()` → per-layer RDMs
4. Compare every layer to the brain RDM (try several metric × method combinations)
5. Stats: mean + 95% CI across model copies; paired tests where models share the same brain RDM

In [7]:
# Untrained models

In [8]:
# Trained models

In [9]:
# YAMNet

In [10]:
# Comparisons, statistics and figures

## Part I — t-SNE of layer activations vs. brain

In [11]:
from sklearn.manifold import TSNE

## Part II (bonus) — own model